Predicting Heart Disease using Machine Learning (KNNeighbours)

The Objectives:
This project will focus on applying and fitting a K neighbours algorithm to discover predictive indicators of heart disease. We will use both supervised and unsupervisoed learning to classify factors and look to better predict rate across a patient population.

The dataset is anonamised from multiple hospitals and includes relevent information for each patient, personal and medical.  There are multiple risk factors attributing to the 17.9 million deaths (WHO estimate) such as unhealthy diet, lack of physical activity or mental illness. Identifying these and other risk factors early on may help prevent many premature deaths.

In [ ]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

EDA: Descriptive Statistics

We will start with exploring our dataset. As per the source, each patient has the following information collected about them:

Age: age of the patient [years]
Sex: sex of the patient [M: Male, F: Female]
ChestPainType: chest pain type [TA: Typical Angina, ATA: Atypical Angina, NAP: Non-Anginal Pain, ASY: Asymptomatic]
RestingBP: resting blood pressure [mm Hg]
Cholesterol: serum cholesterol [mm/dl]
FastingBS: fasting blood sugar [1: if FastingBS > 120 mg/dl, 0: otherwise]
RestingECG: resting electrocardiogram results [Normal: Normal, ST: having ST-T wave abnormality (T wave inversions and/or ST elevation or depression of > 0.05 mV), LVH: showing probable or definite left ventricular hypertrophy by Estes' criteria]
MaxHR: maximum heart rate achieved [Numeric value between 60 and 202]
ExerciseAngina: exercise-induced angina [Y: Yes, N: No]
Oldpeak: oldpeak = ST [Numeric value measured in depression]
ST_Slope: the slope of the peak exercise ST segment [Up: upsloping, Flat: flat, Down: downsloping]
HeartDisease: output class [1: heart disease, 0: Normal]

In [ ]:
#Dataframe:
heart_disease_df = pd.read_csv("heart_disease_prediction.csv")

Exploritary Data Analysis

In [ ]:
#First we take a look at our data:
print(f" We have {heart_disease_df.shape[0]} rows/patients and {heart_disease_df.shape[1]} atributes")
print(heart_disease_df.head(5))

In [ ]:
print(heart_disease_df.dtypes)
heart_disease_df.dtypes.value_counts()

We see that the dataframe contains both numerical and catagorical columns. We also note that FastngBS and HeartDisease have integer type but are categorical variables

Numerical Variables:
Lets first look at our numeric factors

In [ ]:
heart_disease_df.describe()

Firstly we can see that each column count adds up to 918, meaning there are no missing values, just to be sure we check:

In [ ]:
heart_disease_df.isna().sum()

Average age of the patients is 53, and that 55% of our database has heart disease, which is a good proportion for our sample.
Additionally we notice that cholesterol has a median higher than mean, this could mean that it has a slightly rightskewed distribution.

We also spot that some values of the minimum make little sense being 0, RestingBP and Cholesterol are unlikely to be at this value, this could mean that missing values are encoded as 0 for some factors.

As per the dataset description, Cholesterol contains the serum cholesterol values. And per the American Heart Association, serum cholesterol:
is a composite of different measurements. Your total blood cholesterol is calculated by adding your HDL and LDL cholesterol levels, plus 20% of your triglyceride level.
The above information is enough to let us know that a 0 value for serum cholesterol is unlikely.

Categorical Variables:
Now we take a look at our categorical factors

In [ ]:
heart_disease_df.describe(include = 'object')

Observations:
1) Again, seemingly no missing values
2) males have much higher freqency in our sample

In [ ]:
fig1 = plt.figure(figsize=(16,15))

col_names = ["Sex", "ChestPainType", "FastingBS", "RestingECG", "ExerciseAngina", "ST_Slope", "HeartDisease"]
for i in range(1,len(col_names)):
    ax = plt.subplot(4,2,i)
    col = col_names[i]
    plt.bar(heart_disease_df[col].value_counts().index, heart_disease_df[col].value_counts().values)

Data Cleaning

We identified that there are no missing values. However, as we noticed earlier, a couple of columns have 0 values which don't make sense.

We will look at how many zero values RestingBP and Cholesterol contain and decide how to handle those.

In [ ]:
heart_disease_df[heart_disease_df['RestingBP']==0]

In [ ]:
heart_disease_df[heart_disease_df['Cholesterol']==0]

So we only have 1 row of zero for resting Blood pressure and 172 for cholesterol.
This means we can just remove the BP row but given that there are so many rows for Cholesterol we will replace them with the corresponding medians, in relation to the patient having heart disease or not.

In [ ]:
df_clean = heart_disease_df.copy()

# only keep non-zero values for RestingBP
df_clean = df_clean[df_clean["RestingBP"] != 0]

heartdisease_mask = df_clean["HeartDisease"]==0

cholesterol_without_heartdisease = df_clean.loc[heartdisease_mask, "Cholesterol"]
cholesterol_with_heartdisease = df_clean.loc[~heartdisease_mask, "Cholesterol"]

df_clean.loc[heartdisease_mask, "Cholesterol"] = cholesterol_without_heartdisease.replace(to_replace = 0, value = cholesterol_without_heartdisease.median())
df_clean.loc[~heartdisease_mask, "Cholesterol"] = cholesterol_with_heartdisease.replace(to_replace = 0, value = cholesterol_with_heartdisease.median())

Feature Selection

Using our EDA, we have identified some features we can start with:
    
Age
Sex
ChestPainType
Cholesterol
FastingBS

We will also identify how stronly the feature columns are correlated to the target colummn. That should help us narrow down on the features.

In order to do that, we will first convert our categorical columns into dummy variables.

In [ ]:
heart_disease_clean = pd.get_dummies(heart_disease_clean, drop_first=True)

In [ ]:
correlations = abs(heart_disease_clean.corr())

plt.figure(figsize=(12,8))
sns.heatmap(correlations, annot=True, cmap="Greens")

plt.figure(figsize=(12,8))
sns.heatmap(correlations[correlations>0.3], annot=True, cmap='Reds')

From our correlation heatmap, we can identify the following features to be positively correlated (correlation coefficient greater than 0.3) to HeartDisease:

Oldpeak
MaxHR
ChestPainType_ATA
ExerciseAngina_Y
ST_Slope_Flat
ST_Slope_Up
The correlation coefficient threshold was chosen arbitrarily. Surprisingly, Cholesterol is not strongly correlated to HeartDisease. We can consider ignoring the feature for now.

Given everything we have attempted so far, we can narrow down our features to the following:

Oldpeak
Sex_M
It has a relatively low value for the coefficient, but given what we observed in our EDA, let's also take it into account.
ExerciseAngina_Y
ST_Slope_Flat
ST_Slope_Up

In [ ]:
sel = ['Oldpeak', 'Sex_M', 'ExerciseAngina_Y', 'ST_Slope_Flat', 'ST_Slope_Up']

Building a Classifier

First we split our dataset randomly into training, testing and validation sets

In [ ]:
x = heart_disease_clean.drop('HeartDisease', axis = 1)
y = heart_disease_clean['HeartDisease']

X_train, X_test, y_train, y_test = train_test_split(x,y, test_size= 0.10, random_state=1)

X_train, X_val, y_train, y_val = train_test_split(X_train,y_train, test_size= (1/9), random_state=1)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train[sel])
X_val_scaled = scaler.transform(X_val[sel])
X_test_scaled = scaler.transform(X_test[sel])

In [ ]:
grid_params = {"n_neighbors": range(1, 20),
                "metric": ["minkowski", "manhattan"]
              }

knn = KNeighborsClassifier()
knn_grid = GridSearchCV(knn, grid_params, scoring='accuracy')
knn_grid.fit(X_train_scaled, y_train)
knn_grid.best_score_*100, knn_grid.best_params_

In [ ]:
predictions = knn_grid.best_estimator_.predict(X_test_scaled)
accuracy = accuracy_score(y_test, predictions)
print(f" Model Accuracy on test set: {accuracy*100:.2f}")